## Ripple data analysis

Contains Ripple data collected from Gian over two sessions.


In [ ]:
# Loading/Import related packages
import sys
import os

#Usual suspects
import pandas as pd
import numpy as np
import json
import pickle as pkl
import matplotlib.pyplot as plt

#Extras for plotting
from matplotlib.patches import Patch

#Extra for typing
from collections import defaultdict

# Needed to point the importer to the src folder
sys.path.insert(0,r"C:\Users\annas\SynologyDrive\MedUniWien\Projects\NeuroClasp\Students\Liz Kalenteridis\Ripple")


#Decomposition/Processing Imports
from src.muniverse.algorithms.cbss import CBSS

In [3]:
REPO_DIR  = os.path.abspath(os.getcwd())
INPUT_DIR = os.path.join(REPO_DIR, 'data')
OUTPUT_DIR = os.path.join(REPO_DIR, 'results')

combined_data_1 = 'emg_recording_giuan_tewst_20260826_122524.pkl'



In [4]:
def load_simulation_data(input_dir, filename):

    """
    Load the simulation data from a pickle file.

    Parameters:
    - input_dir: str, the directory where the pickle file is located.
    - filename: str, the name of the pickle file.

    Returns:
    - emg_array: dict, the loaded simulation EMG data. 
    Normally contains 320 channels, here downsampled to the first 8x8 array to make decomposition faster .
    """
    file_path = os.path.join(input_dir, filename)

    # Check if the file exists before attempting to load it
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file {file_path} does not exist. Please ensure the input directory and filename are correct.")

    result_pandas = pd.read_pickle(file_path)
    print("Loaded data from:", file_path)
    print(result_pandas.keys())
    print(result_pandas['data'].shape)

    emg_array = result_pandas['data']
    #These two arrays contain the intended poses and durations for each pose
    # We will use this information later in the analysis, see below in the Decomposition Section
    poses = result_pandas['metadata']['poses_intended']
    durations = result_pandas['metadata']['durations']


    #NOTE: a tuple, not a set: a set has no order, so the unpacking below would be random
    return emg_array, poses, durations
    

In [5]:
combined_data_1_path = os.path.join(INPUT_DIR, combined_data_1)
result_pandas = pd.read_pickle(combined_data_1_path)

print(result_pandas.keys())
print(result_pandas['trial_metadata']['items'])

# This task used 3 tasks, each with 8 repitions
n_reps = 8
n_tasks = 3
movement_names = ['fist', 'fast_tripod', 'fast_finger_ext']


dict_keys(['data', 'srate', 'n_channels', 'filtered', 'pre_trigger_seconds', 'timestamp', 'stream_type', 'trial_metadata', 'session_timeline'])
[{'index': 0, 'model': 'fist.glb', 'animation': 0, 'repetitions': 8}, {'index': 1, 'model': 'fasttripodpinch.glb', 'animation': 0, 'repetitions': 8}, {'index': 2, 'model': 'fastfingerext.glb', 'animation': 0, 'repetitions': 8}]


In [6]:
session1_folder = os.path.join(INPUT_DIR, 'emg_recording_giuan_tewst_20260826_122524_divided')
session2_folder = os.path.join(INPUT_DIR, 'emg_recording_giuan_tewst2_20260826_135844_divided')



In [7]:
import glob 

def get_task_data(input_dir, n_reps):
    """
    Concatenate EMG data from multiple pickle files.

    Parameters:
    - pckl_files: list of str, the names of the pickle files to concatenate.
    - input_dir: str, the directory where the pickle files are located.

    Returns:
    - concatenated_data: np.ndarray, the concatenated EMG data.
    """
    move_files =  sorted(glob.glob(os.path.join(input_dir + '*/move*')))

    movement_dict= {}
    for movement_num, movement_type in enumerate(movement_names):
        start = movement_num * n_reps
        end = start + n_reps
        task_files = move_files[start:end]

        emg_data = [pd.DataFrame(pd.read_pickle(file_name)['data']) for file_name in task_files]

        duration = [pd.read_pickle(file_name)['duration_s'] for file_name in task_files]

        movement_dict[movement_type] = {
            'emg_data_list': emg_data, # list containing EMG data, length = reps
            'movement_dur': duration # list containing all durations for each rep
        }

    return movement_dict

def get_iso(movement_data, sampling_freq):
    """
    Isolate the middle 4 seconds of each movement, during which the movement is held isometrically

    Paramters:
    - movement_data: dictionary containing keys corresponding to movement types.
        Each key contains 'emg_data' and 'movement_dur' corresponding to emg data and durations (in s) for each repition within the movement type.
    - sampling_freq: float, sampling frequency of data collection in Hz

    Returns:
    - iso_dict: dict containing same keys as movement_data but with 'emg_data_list' from the middle of the movement duration +/- 2 seconds, total duration is 4 seconds of isometric movement
    """

    iso_dict = {}
    for movement in movement_data.keys():
        emg_reps = movement_data[movement]['emg_data_list']
        duration_reps = movement_data[movement]['movement_dur']

        emg_iso_data = []
        for emg_data, dur in zip(emg_reps, duration_reps):
            iso_start = int(((dur/2)-2) * sampling_freq)
            iso_end = int(((dur/2)+2) * sampling_freq)

            emg_iso = emg_data.iloc[:, iso_start:iso_end]
            emg_iso_data.append(emg_iso)

        combined_iso_emg = pd.concat(emg_iso_data, ignore_index = True, axis = 1)

        iso_dict[movement]= {
            'emg_iso_reps': emg_iso_data,
            'emg_iso_combined': combined_iso_emg
        }

    return(iso_dict)

In [8]:
session1_data = get_task_data(session1_folder, 8)
session1_iso = get_iso(session1_data, 2000)
print(session1_iso.keys())
print(session1_iso['fist']['emg_iso_combined'].shape)
print(session1_iso['fist']['emg_iso_reps'][0].shape)


dict_keys(['fist', 'fast_tripod', 'fast_finger_ext'])
(32, 64001)
(32, 8001)


In [13]:
session1_fist_emg = session1_iso['fist']['emg_iso_combined'].to_numpy()
print(session1_fist_emg.shape)
session1_ft_emg = session1_iso['fast_tripod']['emg_iso_combined'].to_numpy()
session1_ffe_emg = session1_iso['fast_finger_ext']['emg_iso_combined'].to_numpy()

(32, 64001)


In [ ]:
#Important: if you are in a jupyter notebook,
#this line allows the plots to be displayed in a separate window instead of inline
#which is needed for the mask and channel reviews gui
%matplotlib qt


# sys.path.append(os.path.abspath('/Users/lizkal/Library/CloudStorage/SynologyDrive-Personal/MotorUnitSuite/src'))

from src.utils.select_windows import select_windows
from src.utils.review_channels import review_channels

In [11]:
good_mask_path = "demo_good_mask.npy"
mask_path = "demo_mask.npy"
exclude_path = "demo_exclude.npy"

In [ ]:
review_channels(session1_fist_emg, good_mask_path, fs=2000, label="demo")

seed: auto RMS-outlier mask (32/32 good, bad [])


In [15]:
from scipy.interpolate import interp1d

def remove_spikes(emg, threshold_std=5):
    """
    Remove large spikes via simple thresholding and linear interpolation.
    
    Parameters
    ----------
    emg : ndarray, shape (n_channels, n_samples)
    threshold_std : float
        Number of standard deviations for threshold
    
    Returns
    -------
    emg_clean : ndarray
    spike_mask : ndarray, bool
    """
    emg_clean = emg.copy()
    spike_mask = np.zeros_like(emg, dtype=bool)
    
    for ch in range(emg.shape[0]):
        signal = emg[ch]
        threshold = threshold_std * np.std(signal)
        
        spikes = np.abs(signal) > threshold
        spike_mask[ch] = spikes
        
        if np.any(spikes):
            clean_idx = np.where(~spikes)[0]
            spike_idx = np.where(spikes)[0]
            
            if len(clean_idx) > 1:
                interp = interp1d(clean_idx, signal[clean_idx], 
                                  kind='linear', bounds_error=False, 
                                  fill_value='extrapolate')
                emg_clean[ch, spike_idx] = interp(spike_idx)
    
    return emg_clean, spike_mask


In [18]:
session1_fist_clean, spikes = remove_spikes(session1_fist_emg, threshold_std=3)


In [19]:
review_channels(session1_fist_clean, good_mask_path, fs=2000, label="demo")

seed: auto RMS-outlier mask (32/32 good, bad [])
